# Fresh vs Rotten Fruit Classification — MobileNetV2

## 1. MobileNetV2 Transfer Learning

This notebook implements MobileNetV2 transfer learning for six-class fresh-versus-rotten fruit classification.

The model uses the same verified leak-free train, validation, and test splits as the Baseline CNN so that the two models can be compared fairly.

The work in this notebook is limited to MobileNetV2 transfer learning and experimentation.

## 2. Reproducibility and Configuration

The MobileNetV2 experiments use a fixed random seed and the same image size and class order as the verified Baseline CNN.

The official dataset split is:
- Training: 9,513 images
- Validation: 2,043 images
- Test: 2,043 images

The six classes are kept in the same order throughout the project.

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

NUM_CLASSES = len(CLASS_NAMES)

print("TensorFlow version:", tf.__version__)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)

## 3. Load the Official Leak-Free Dataset Splits

The MobileNetV2 model uses the same verified train, validation, and test CSV files as the Baseline CNN.

This keeps the model comparison fair and preserves the source-group-aware split created during preprocessing.

In [ ]:
import pandas as pd
from pathlib import Path

GITHUB_RAW = (
    "https://raw.githubusercontent.com/"
    "onkar-726/fresh-rotten-fruit-classification/main/data"
)

train_df = pd.read_csv(f"{GITHUB_RAW}/train.csv")
validation_df = pd.read_csv(f"{GITHUB_RAW}/validation.csv")
test_df = pd.read_csv(f"{GITHUB_RAW}/test.csv")

print("Training rows  :", len(train_df))
print("Validation rows:", len(validation_df))
print("Testing rows  :", len(test_df))
print("Total rows     :", len(train_df) + len(validation_df) + len(test_df))

print("\nColumns:")
print(train_df.columns.tolist())

## 4. Connect the Dataset Images

The CSV files contain relative image paths from the Kaggle dataset.

The actual dataset is downloaded into the current Colab runtime, and the relative paths are joined with the dataset root.

The paths are verified before building the MobileNetV2 input pipeline.

In [ ]:
import kagglehub
from pathlib import Path

# Download the Kaggle dataset into the Colab runtime.
dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)

DATASET_ROOT = Path(dataset_path) / "dataset"

print("Dataset root:", DATASET_ROOT)
print("Dataset root exists:", DATASET_ROOT.exists())

# Check that the CSV paths point to real image files.
for name, dataframe in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Testing", test_df),
]:
    full_paths = dataframe["image_path"].apply(
        lambda path: DATASET_ROOT / path
    )

    missing_count = (~full_paths.apply(Path.exists)).sum()

    print(f"{name} missing images:", missing_count)

print("\nSample image path:")
sample_relative_path = train_df.iloc[0]["image_path"]
sample_full_path = DATASET_ROOT / sample_relative_path

print("Relative:", sample_relative_path)
print("Full    :", sample_full_path)
print("Exists  :", sample_full_path.exists())

## 5. MobileNetV2 Image Preprocessing

MobileNetV2 uses its application-specific preprocessing function instead of the 0–1 normalization used by the Baseline CNN.

The preprocessing converts images to RGB, resizes them to 224 × 224 pixels, converts them to float32, and applies the MobileNetV2 preprocessing function.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


def preprocess_mobilenet_image(image_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image

In [ ]:
sample_relative_path = train_df.iloc[0]["image_path"]
sample_full_path = DATASET_ROOT / sample_relative_path

processed_image = preprocess_mobilenet_image(
    tf.constant(str(sample_full_path))
)

print("Processed image shape:", processed_image.shape)
print("Processed image dtype:", processed_image.dtype)
print("Minimum value:", tf.reduce_min(processed_image).numpy())
print("Maximum value:", tf.reduce_max(processed_image).numpy())

## 6. Training-Time Data Augmentation

Training-time augmentation is used to create small variations of training images without creating additional image files on disk.

The augmentation is applied only to training data. Validation and test images remain unchanged apart from the required MobileNetV2 preprocessing.

The augmentation includes horizontal flipping, small rotations, zoom changes, brightness changes, and contrast changes.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=IMAGE_SIZE + (3,)
        ),
        tf.keras.layers.RandomFlip(
            mode="horizontal"
        ),
        tf.keras.layers.RandomRotation(
            factor=0.05
        ),
        tf.keras.layers.RandomZoom(
            height_factor=0.10,
            width_factor=0.10
        ),
        tf.keras.layers.RandomBrightness(
            factor=0.10,
            value_range=(0, 255)
        ),
        tf.keras.layers.RandomContrast(
            factor=0.10
        ),
    ],
    name="mobileNet_training_augmentation"
)

print("Training-time augmentation created.")
print(data_augmentation.name)

In [ ]:
# Take one real training image.
sample_relative_path = train_df.iloc[0]["image_path"]
sample_full_path = DATASET_ROOT / sample_relative_path

# Load image in the same 0–255 range expected by the augmentation layers.
image_bytes = tf.io.read_file(str(sample_full_path))
sample_image = tf.image.decode_image(
    image_bytes,
    channels=3,
    expand_animations=False
)

sample_image = tf.image.resize(sample_image, IMAGE_SIZE)
sample_image = tf.cast(sample_image, tf.float32)

# Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
sample_batch = tf.expand_dims(sample_image, axis=0)

# Apply training augmentation.
augmented_batch = data_augmentation(sample_batch, training=True)

print("Original shape :", sample_batch.shape)
print("Augmented shape:", augmented_batch.shape)
print("Original dtype :", sample_batch.dtype)
print("Augmented dtype:", augmented_batch.dtype)
print(
    "Augmented range:",
    float(tf.reduce_min(augmented_batch).numpy()),
    "to",
    float(tf.reduce_max(augmented_batch).numpy())
)

## 7. MobileNetV2 Training Input Pipeline

The input pipeline converts the verified CSV image paths into TensorFlow datasets.

Training images use the verified training-time augmentation before MobileNetV2 preprocessing.

Validation and test images are not augmented. They only undergo image decoding, resizing, and MobileNetV2 preprocessing.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}


def load_mobilenet_image(image_path, label, training=False):
    # Read image file.
    image_bytes = tf.io.read_file(image_path)

    # Decode as RGB.
    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    # Resize to the MobileNetV2 input size.
    image = tf.image.resize(
        image,
        IMAGE_SIZE
    )

    # Convert to float32.
    image = tf.cast(
        image,
        tf.float32
    )

    if training:
        # The Keras augmentation Sequential model expects
        # batched input, so temporarily add a batch dimension.
        image = tf.expand_dims(
            image,
            axis=0
        )

        image = data_augmentation(
            image,
            training=True
        )

        # Remove the temporary batch dimension.
        image = tf.squeeze(
            image,
            axis=0
        )

    # Keep the image in the valid 0–255 range
    # before applying MobileNetV2 preprocessing.
    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    # MobileNetV2 preprocessing: approximately 0–255 -> -1 to +1.
    image = preprocess_input(image)

    return image, label


def build_mobilenet_dataset(dataframe, training=False):
    # Convert relative CSV paths to full Colab paths.
    image_paths = [
        str(DATASET_ROOT / path)
        for path in dataframe["image_path"]
    ]

    # Convert class names to integer labels.
    labels = [
        CLASS_TO_INDEX[class_name]
        for class_name in dataframe["class_name"]
    ]

    # Create TensorFlow dataset from paths and labels.
    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    # Shuffle only the training dataset.
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    # Load and preprocess images.
    dataset = dataset.map(
        lambda image_path, label: load_mobilenet_image(
            image_path,
            label,
            training=training
        ),
        num_parallel_calls=AUTOTUNE
    )

    # Create batches.
    dataset = dataset.batch(
        BATCH_SIZE
    )

    # Improve input pipeline performance.
    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset


# Create the three official datasets.
train_ds = build_mobilenet_dataset(
    train_df,
    training=True
)

validation_ds = build_mobilenet_dataset(
    validation_df,
    training=False
)

test_ds = build_mobilenet_dataset(
    test_df,
    training=False
)

print("MobileNetV2 datasets created.")

In [ ]:
train_batch_images, train_batch_labels = next(
    iter(train_ds)
)

validation_batch_images, validation_batch_labels = next(
    iter(validation_ds)
)

test_batch_images, test_batch_labels = next(
    iter(test_ds)
)


print(
    "Training image batch shape:",
    train_batch_images.shape
)

print(
    "Training label batch shape:",
    train_batch_labels.shape
)

print(
    "Validation image batch shape:",
    validation_batch_images.shape
)

print(
    "Validation label batch shape:",
    validation_batch_labels.shape
)

print(
    "Test image batch shape:",
    test_batch_images.shape
)

print(
    "Test label batch shape:",
    test_batch_labels.shape
)


print(
    "\nTraining pixel range:",
    float(tf.reduce_min(train_batch_images).numpy()),
    "to",
    float(tf.reduce_max(train_batch_images).numpy())
)

print(
    "Validation pixel range:",
    float(tf.reduce_min(validation_batch_images).numpy()),
    "to",
    float(tf.reduce_max(validation_batch_images).numpy())
)

print(
    "Test pixel range:",
    float(tf.reduce_min(test_batch_images).numpy()),
    "to",
    float(tf.reduce_max(test_batch_images).numpy())
)


print(
    "\nTraining labels range:",
    int(tf.reduce_min(train_batch_labels).numpy()),
    "to",
    int(tf.reduce_max(train_batch_labels).numpy())
)

## 8. Build the MobileNetV2 Transfer-Learning Model

MobileNetV2 is initialized with ImageNet-pretrained weights and its original classification head is removed.

A new classification head is added for the six fruit classes.

The MobileNetV2 feature extractor is initially frozen so that only the new classification head is trained in Stage 1.

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2


base_model = MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)

# Freeze the pretrained feature extractor for Stage 1.
base_model.trainable = False


inputs = layers.Input(
    shape=IMAGE_SIZE + (3,),
    name="image_input"
)

# The input has already been preprocessed by the tf.data pipeline.
x = base_model(
    inputs,
    training=False
)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = layers.Dropout(
    0.20,
    name="dropout"
)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classifier"
)(x)


mobilenet_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="MobileNetV2_Fruit_Classifier"
)


print("MobileNetV2 model created.")
print("Base model trainable:", base_model.trainable)
print("Total parameters:", mobilenet_model.count_params())

mobilenet_model.summary()

## 9. MobileNetV2 Forward-Pass Test

A real batch from the training dataset is passed through the MobileNetV2 model before training.

This verifies that the input pipeline, MobileNetV2 preprocessing, model input shape, and six-class output are connected correctly.

In [ ]:
# Take one real batch from the training dataset.
sample_images, sample_labels = next(
    iter(train_ds)
)

# Run a forward pass without training.
sample_predictions = mobilenet_model(
    sample_images,
    training=False
)

print("Input batch shape:", sample_images.shape)
print("Label batch shape:", sample_labels.shape)
print("Output shape:", sample_predictions.shape)

print(
    "First prediction probabilities:",
    sample_predictions[0].numpy()
)

print(
    "First prediction probability sum:",
    float(tf.reduce_sum(sample_predictions[0]).numpy())
)

print(
    "Predicted class index for first image:",
    int(tf.argmax(sample_predictions[0]).numpy())
)

## 10. Compile MobileNetV2 for Stage 1

In Stage 1, the pretrained MobileNetV2 feature extractor remains frozen.

Only the newly added six-class classification head will be trained.

Adam is used with a learning rate of 0.001, matching the optimizer setup used for the Baseline CNN's initial training.

In [ ]:
import numpy as np
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy


# Stage 1: keep the pretrained MobileNetV2 backbone frozen.
base_model.trainable = False

mobilenet_model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss=SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

trainable_parameter_count = sum(
    np.prod(weight.shape)
    for weight in mobilenet_model.trainable_weights
)

print("MobileNetV2 Stage 1 compiled successfully.")
print("Base model trainable:", base_model.trainable)
print("Learning rate:", 0.001)
print("Trainable parameters:", trainable_parameter_count)

## 11. MobileNetV2 Stage 1 Training Configuration

Stage 1 trains only the newly added classification head while the pretrained MobileNetV2 backbone remains frozen.

Class weights are calculated from the training split to reduce the effect of class imbalance.

Early stopping, learning-rate reduction, and best-model checkpointing are used to control training and retain the best validation result.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


# Calculate class weights using only the training split.
train_class_indices = np.array([
    CLASS_TO_INDEX[class_name]
    for class_name in train_df["class_name"]
])

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_class_indices
)

class_weights = {
    class_index: weight
    for class_index, weight in enumerate(class_weight_values)
}

print("Class weights:")

for class_index, class_name in enumerate(CLASS_NAMES):
    print(
        f"{class_index} ({class_name}): "
        f"{class_weights[class_index]:.4f}"
    )


STAGE1_EPOCHS = 10

stage1_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/content/mobilenetv2_stage1_best.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]


print("\nStage 1 configuration:")
print("Epoch limit:", STAGE1_EPOCHS)
print("Callbacks:", len(stage1_callbacks))
print("Backbone trainable:", base_model.trainable)
print("Checkpoint:", "/content/mobilenetv2_stage1_best.keras")

## 12. MobileNetV2 Stage 1 Training

The MobileNetV2 backbone remains frozen during Stage 1.

Only the newly added classification head is trained using the full leak-free training split.

Training-time augmentation and class weights are enabled. The validation split is used to monitor generalization.

In [ ]:
history_stage1 = mobilenet_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=STAGE1_EPOCHS,
    class_weight=class_weights,
    callbacks=stage1_callbacks,
    verbose=1
)

print("MobileNetV2 Stage 1 training completed.")

## 13. Load the Best Stage 1 Checkpoint

The best Stage 1 model is restored from the checkpoint with the lowest validation loss.

Stage 2 fine-tuning will start from these best Stage 1 weights rather than from the final Epoch 10 weights.

In [ ]:
from pathlib import Path

BEST_STAGE1_PATH = Path(
    "/content/mobilenetv2_stage1_best.keras"
)

print(
    "Checkpoint exists:",
    BEST_STAGE1_PATH.exists()
)

# Load the best Stage 1 model.
mobilenet_model = tf.keras.models.load_model(
    BEST_STAGE1_PATH
)

# Retrieve the pretrained MobileNetV2 backbone.
base_model = mobilenet_model.get_layer(
    "mobilenetv2_1.00_224"
)

print("Best Stage 1 model loaded successfully.")
print("Base model trainable:", base_model.trainable)
print("Total parameters:", mobilenet_model.count_params())

trainable_parameter_count = sum(
    np.prod(weight.shape)
    for weight in mobilenet_model.trainable_weights
)

print(
    "Trainable parameters:",
    trainable_parameter_count
)

## 14. Prepare MobileNetV2 for Stage 2 Fine-Tuning

Stage 2 fine-tunes the upper part of the pretrained MobileNetV2 feature extractor.

The last 30 layers are selected for fine-tuning, while BatchNormalization layers remain frozen. Keeping BatchNormalization layers frozen helps preserve the pretrained normalization behavior during fine-tuning.

The best Stage 1 checkpoint loaded in the previous section is used as the starting point.

In [ ]:
from tensorflow.keras.layers import BatchNormalization


# Start with the complete MobileNetV2 backbone frozen.
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = False


# Fine-tune only the last 30 layers, excluding BatchNormalization layers.
FINE_TUNE_LAYERS = 30

for layer in base_model.layers[-FINE_TUNE_LAYERS:]:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = True


# Keep the backbone in inference mode so BatchNormalization layers
# continue using their stored moving statistics.
# This is also consistent with the way the base model was called
# when the classifier was constructed.
base_model.trainable = True


trainable_layers = sum(
    layer.trainable
    for layer in base_model.layers
)

frozen_layers = len(base_model.layers) - trainable_layers

trainable_parameter_count = sum(
    np.prod(weight.shape)
    for weight in mobilenet_model.trainable_weights
)

batch_norm_trainable = sum(
    layer.trainable
    for layer in base_model.layers
    if isinstance(layer, BatchNormalization)
)

print("MobileNetV2 Stage 2 fine-tuning layers prepared.")
print("Total backbone layers:", len(base_model.layers))
print("Trainable backbone layers:", trainable_layers)
print("Frozen backbone layers:", frozen_layers)
print("Trainable BatchNormalization layers:", batch_norm_trainable)
print("Total trainable parameters:", trainable_parameter_count)

## 15. Compile MobileNetV2 for Stage 2 Fine-Tuning

Stage 2 fine-tunes the selected upper MobileNetV2 layers together with the new classification head.

A small learning rate of 1e-5 is used so that the pretrained features are adjusted gradually rather than changed aggressively.

BatchNormalization layers remain frozen during fine-tuning.

In [ ]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy


STAGE2_LEARNING_RATE = 1e-5

mobilenet_model.compile(
    optimizer=Adam(
        learning_rate=STAGE2_LEARNING_RATE
    ),
    loss=SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

trainable_parameter_count = sum(
    np.prod(weight.shape)
    for weight in mobilenet_model.trainable_weights
)

trainable_batch_norm_count = sum(
    layer.trainable
    for layer in base_model.layers
    if isinstance(layer, tf.keras.layers.BatchNormalization)
)

print("MobileNetV2 Stage 2 compiled successfully.")
print("Learning rate:", STAGE2_LEARNING_RATE)
print("Base model trainable:", base_model.trainable)
print("Trainable BatchNormalization layers:", trainable_batch_norm_count)
print("Total trainable parameters:", trainable_parameter_count)